# Frozen DistilBERT Router: Two-Head Depth/Width Classifier With Training-Fold Oversampling

This notebook trains a Router that maps a query to two positive integers: search depth and beam width.

Configuration:
- Model: `distilbert-base-uncased`
- Encoder: frozen
- Trainable layers: dropout + depth classifier head + width classifier head
- Max sequence length: 64
- Batch size: 16
- Epochs: 30
- Learning rate: 1e-3
- Weight decay: 0.01
- Dropout: 0.2
- Scheduler: none
- Loss: unweighted cross entropy for depth + unweighted cross entropy for width
- Evaluation: 5-fold CV, roughly 80/20 per fold
- Oversampling: training fold only, by full strategy tuple, capped at `oversample_target_per_strategy`


In [2]:
# If needed, install dependencies first:
# !pip install torch transformers scikit-learn pandas tqdm

import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

In [3]:
CONFIG = {
    "data_path": "/content/clean_router_training_data.jsonl",
    "model_name": "distilbert-base-uncased",
    "max_length": 64,
    "batch_size": 16,
    "epochs": 30,
    "learning_rate": 1e-3,
    "weight_decay": 0.01,
    "dropout": 0.2,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "oversample_target_per_strategy": 10,
    "seed": 42,
}

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='cuda')

## Load Data

Expected JSONL format:

```json
{"query": "...", "strategy": "(2,3)"}
```

In [4]:
def parse_strategy(strategy: str) -> tuple[int, int]:
    strategy = strategy.strip()
    if not (strategy.startswith("(") and strategy.endswith(")")):
        raise ValueError(f"Bad strategy format: {strategy}")
    left, right = strategy[1:-1].split(",")
    return int(left), int(right)

data_path = Path(CONFIG["data_path"])
rows = []
with data_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        depth, width = parse_strategy(obj["strategy"])
        rows.append({
            "query": obj["query"],
            "strategy": obj["strategy"],
            "depth": depth,
            "width": width,
        })

df = pd.DataFrame(rows)

depth_values = sorted(df["depth"].unique())
width_values = sorted(df["width"].unique())
depth2id = {value: idx for idx, value in enumerate(depth_values)}
id2depth = {idx: value for value, idx in depth2id.items()}
width2id = {value: idx for idx, value in enumerate(width_values)}
id2width = {idx: value for value, idx in width2id.items()}

df["depth_label"] = df["depth"].map(depth2id)
df["width_label"] = df["width"].map(width2id)

print(f"Examples: {len(df)}")
print(f"Depth classes: {depth_values}")
print(f"Width classes: {width_values}")
display(df.head())
display(df["strategy"].value_counts().rename_axis("strategy").reset_index(name="count"))
display(df["depth"].value_counts().sort_index().rename_axis("depth").reset_index(name="count"))
display(df["width"].value_counts().sort_index().rename_axis("width").reset_index(name="count"))

Examples: 127
Depth classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Width classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(5)]


,query,strategy,depth,width,depth_label,width_label
0,what continent is australia in,"(1,2)",1,2,0,1
1,where does missouri river end,"(1,1)",1,1,0,0
2,who wrote the book of st. john,"(1,1)",1,1,0,0
3,what illness does michael j fox have,"(1,1)",1,1,0,0
4,what did john howard study at university,"(4,1)",4,1,3,0


,strategy,count
0,"(1,1)",43
1,"(2,1)",20
2,"(1,2)",13
3,"(3,1)",10
4,"(2,3)",7
5,"(2,5)",7
6,"(1,3)",4
7,"(4,1)",4
8,"(1,5)",4
9,"(4,3)",4


,depth,count
0,1,64
1,2,37
2,3,14
3,4,12


,width,count
0,1,77
1,2,18
2,3,17
3,5,15


## Folds

This splitter distributes full strategy labels across folds as evenly as possible, without validation leakage. The model predicts depth and width separately.

Oversampling is applied later **inside each training fold only**. The validation fold stays unchanged.


In [5]:
def make_stratified_like_folds(labels, n_splits: int, seed: int):
    rng = np.random.default_rng(seed)
    label_to_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        label_to_indices[label].append(idx)

    folds = [[] for _ in range(n_splits)]
    for offset, label in enumerate(sorted(label_to_indices)):
        indices = np.array(label_to_indices[label])
        rng.shuffle(indices)
        for position, idx in enumerate(indices):
            folds[(offset + position) % n_splits].append(int(idx))

    all_indices = set(range(len(labels)))
    splits = []
    for val_indices in folds:
        val_indices = sorted(val_indices)
        train_indices = sorted(all_indices - set(val_indices))
        splits.append((train_indices, val_indices))
    return splits

splits = make_stratified_like_folds(df["strategy"].tolist(), CONFIG["n_splits"], CONFIG["seed"])
for fold_id, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"Fold {fold_id}: train={len(train_idx)}, val={len(val_idx)}")
    print("  val strategy distribution:", dict(Counter(df.iloc[val_idx]["strategy"])))
    print("  val depth distribution:", dict(Counter(df.iloc[val_idx]["depth"])))
    print("  val width distribution:", dict(Counter(df.iloc[val_idx]["width"])))

Fold 1: train=101, val=26
  val strategy distribution: {'(1,2)': 2, '(1,1)': 9, '(3,1)': 2, '(1,3)': 1, '(2,1)': 4, '(2,2)': 1, '(4,5)': 1, '(4,1)': 1, '(2,3)': 1, '(1,5)': 1, '(2,5)': 1, '(3,3)': 1, '(4,3)': 1}
  val depth distribution: {1: 13, 3: 3, 2: 7, 4: 3}
  val width distribution: {2: 3, 1: 16, 3: 4, 5: 3}
Fold 2: train=100, val=27
  val strategy distribution: {'(2,3)': 2, '(4,3)': 1, '(1,2)': 3, '(3,5)': 1, '(1,1)': 9, '(1,5)': 1, '(3,1)': 2, '(2,1)': 4, '(2,5)': 1, '(3,3)': 1, '(2,2)': 1, '(4,5)': 1}
  val depth distribution: {2: 8, 4: 2, 1: 13, 3: 4}
  val width distribution: {3: 4, 2: 4, 5: 4, 1: 15}
Fold 3: train=100, val=27
  val strategy distribution: {'(1,1)': 9, '(4,1)': 1, '(3,1)': 2, '(2,3)': 2, '(2,1)': 4, '(1,2)': 3, '(2,5)': 2, '(2,2)': 1, '(1,3)': 1, '(4,5)': 1, '(4,3)': 1}
  val depth distribution: {1: 13, 4: 3, 3: 2, 2: 9}
  val width distribution: {1: 16, 3: 4, 2: 4, 5: 3}
Fold 4: train=103, val=24
  val strategy distribution: {'(1,1)': 8, '(3,1)': 2, '(1,2)':

## Training-Fold Oversampling

Oversampling is done by full strategy tuple after each train/validation split. This balances both heads together while keeping validation honest.


In [6]:
def oversample_by_strategy(frame: pd.DataFrame, target_count: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    pieces = []
    for strategy, group in frame.groupby("strategy", sort=True):
        pieces.append(group)
        needed = target_count - len(group)
        if needed > 0:
            sampled_positions = rng.choice(len(group), size=needed, replace=True)
            pieces.append(group.iloc[sampled_positions])

    oversampled = pd.concat(pieces, ignore_index=True)
    oversampled = oversampled.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return oversampled


example_oversampled = oversample_by_strategy(
    df, CONFIG["oversample_target_per_strategy"], CONFIG["seed"]
)
print("Original examples:", len(df))
print("Oversampled example size if applied to all data:", len(example_oversampled))
display(example_oversampled["strategy"].value_counts().rename_axis("strategy").reset_index(name="count"))


Original examples: 127
Oversampled example size if applied to all data: 206


,strategy,count
0,"(1,1)",43
1,"(2,1)",20
2,"(1,2)",13
3,"(4,5)",10
4,"(2,2)",10
5,"(4,2)",10
6,"(3,5)",10
7,"(1,5)",10
8,"(3,3)",10
9,"(1,3)",10


## Dataset and Model

In [7]:
class RouterDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_length: int):
        self.queries = frame["query"].tolist()
        self.depth_labels = frame["depth_label"].astype(int).tolist()
        self.width_labels = frame["width_label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.queries[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "depth_labels": torch.tensor(self.depth_labels[idx], dtype=torch.long),
            "width_labels": torch.tensor(self.width_labels[idx], dtype=torch.long),
        }


class FrozenDistilBertTwoHeadRouter(nn.Module):
    def __init__(self, model_name: str, num_depth_labels: int, num_width_labels: int, dropout: float):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.depth_classifier = nn.Linear(hidden_size, num_depth_labels)
        self.width_classifier = nn.Linear(hidden_size, num_width_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(cls_embedding)
        depth_logits = self.depth_classifier(pooled)
        width_logits = self.width_classifier(pooled)
        return depth_logits, width_logits

## Training and Evaluation Helpers

In [8]:
def train_one_epoch(model, loader, optimizer, depth_loss_fn, width_loss_fn):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        depth_labels = batch["depth_labels"].to(device)
        width_labels = batch["width_labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        depth_logits, width_logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = depth_loss_fn(depth_logits, depth_labels) + width_loss_fn(width_logits, width_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * depth_labels.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, depth_loss_fn, width_loss_fn):
    model.eval()
    total_loss = 0.0
    all_depth_preds = []
    all_width_preds = []
    all_depth_labels = []
    all_width_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        depth_labels = batch["depth_labels"].to(device)
        width_labels = batch["width_labels"].to(device)

        depth_logits, width_logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = depth_loss_fn(depth_logits, depth_labels) + width_loss_fn(width_logits, width_labels)

        depth_preds = depth_logits.argmax(dim=-1)
        width_preds = width_logits.argmax(dim=-1)
        total_loss += loss.item() * depth_labels.size(0)

        all_depth_preds.extend(depth_preds.cpu().tolist())
        all_width_preds.extend(width_preds.cpu().tolist())
        all_depth_labels.extend(depth_labels.cpu().tolist())
        all_width_labels.extend(width_labels.cpu().tolist())

    metrics = compute_metrics(all_depth_labels, all_width_labels, all_depth_preds, all_width_preds)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics, all_depth_labels, all_width_labels, all_depth_preds, all_width_preds


def compute_metrics(depth_labels, width_labels, depth_preds, width_preds):
    gold_depth = [id2depth[i] for i in depth_labels]
    pred_depth = [id2depth[i] for i in depth_preds]
    gold_width = [id2width[i] for i in width_labels]
    pred_width = [id2width[i] for i in width_preds]

    exact = [gd == pd and gw == pw for gd, gw, pd, pw in zip(gold_depth, gold_width, pred_depth, pred_width)]
    gold_strategy = [f"({d},{w})" for d, w in zip(gold_depth, gold_width)]
    pred_strategy = [f"({d},{w})" for d, w in zip(pred_depth, pred_width)]

    return {
        "exact_accuracy": float(np.mean(exact)),
        "depth_accuracy": accuracy_score(gold_depth, pred_depth),
        "width_accuracy": accuracy_score(gold_width, pred_width),
        "depth_macro_f1": f1_score(gold_depth, pred_depth, average="macro", zero_division=0),
        "width_macro_f1": f1_score(gold_width, pred_width, average="macro", zero_division=0),
        "strategy_macro_f1": f1_score(gold_strategy, pred_strategy, average="macro", zero_division=0),
        "depth_abs_error": np.mean([abs(a - b) for a, b in zip(gold_depth, pred_depth)]),
        "width_abs_error": np.mean([abs(a - b) for a, b in zip(gold_width, pred_width)]),
        "cost_sensitive_error": np.mean([
            abs(gd - pd) + 0.5 * abs(gw - pw)
            for gd, gw, pd, pw in zip(gold_depth, gold_width, pred_depth, pred_width)
        ]),
    }

## Majority Baselines

In [9]:
majority_depth = df["depth"].mode()[0]
majority_width = df["width"].mode()[0]
majority_strategy = df["strategy"].mode()[0]

print("Majority depth:", majority_depth)
print("Majority width:", majority_width)
print("Majority full strategy:", majority_strategy)
print("Always majority full strategy exact accuracy:", (df["strategy"] == majority_strategy).mean())
print("Always majority-depth/majority-width exact accuracy:", ((df["depth"] == majority_depth) & (df["width"] == majority_width)).mean())
print("Always majority depth accuracy:", (df["depth"] == majority_depth).mean())
print("Always majority width accuracy:", (df["width"] == majority_width).mean())

Majority depth: 1
Majority width: 1
Majority full strategy: (1,1)
Always majority full strategy exact accuracy: 0.33858267716535434
Always majority-depth/majority-width exact accuracy: 0.33858267716535434
Always majority depth accuracy: 0.5039370078740157
Always majority width accuracy: 0.6062992125984252


## 5-Fold Cross-Validation

In [10]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

fold_results = []
all_predictions = []

for fold_id, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"\n===== Fold {fold_id}/{CONFIG['n_splits']} =====")
    set_seed(CONFIG["seed"] + fold_id)

    train_df_raw = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    train_df = oversample_by_strategy(
        train_df_raw,
        target_count=CONFIG["oversample_target_per_strategy"],
        seed=CONFIG["seed"] + fold_id,
    )
    print(f"Raw train size: {len(train_df_raw)} | Oversampled train size: {len(train_df)} | Val size: {len(val_df)}")
    print("Oversampled train strategy distribution:", dict(Counter(train_df["strategy"])))

    train_dataset = RouterDataset(train_df, tokenizer, CONFIG["max_length"])
    val_dataset = RouterDataset(val_df, tokenizer, CONFIG["max_length"])
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

    model = FrozenDistilBertTwoHeadRouter(
        CONFIG["model_name"],
        num_depth_labels=len(depth_values),
        num_width_labels=len(width_values),
        dropout=CONFIG["dropout"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
    )
    depth_loss_fn = nn.CrossEntropyLoss()
    width_loss_fn = nn.CrossEntropyLoss()

    best_val_loss = math.inf
    best_state = None
    best_epoch = 0
    patience_used = 0

    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, depth_loss_fn, width_loss_fn)
        val_metrics, _, _, _, _ = evaluate(model, val_loader, depth_loss_fn, width_loss_fn)

        print(
            f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"exact_acc={val_metrics['exact_accuracy']:.3f} | "
            f"depth_acc={val_metrics['depth_accuracy']:.3f} | "
            f"width_acc={val_metrics['width_accuracy']:.3f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            patience_used = 0
        else:
            patience_used += 1
            if patience_used >= CONFIG["early_stopping_patience"]:
                print(f"Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    final_metrics, depth_labels, width_labels, depth_preds, width_preds = evaluate(
        model, val_loader, depth_loss_fn, width_loss_fn
    )
    final_metrics["fold"] = fold_id
    final_metrics["best_epoch"] = best_epoch
    fold_results.append(final_metrics)

    for row, depth_gold_id, width_gold_id, depth_pred_id, width_pred_id in zip(
        val_df.to_dict("records"), depth_labels, width_labels, depth_preds, width_preds
    ):
        gold_depth = id2depth[depth_gold_id]
        gold_width = id2width[width_gold_id]
        pred_depth = id2depth[depth_pred_id]
        pred_width = id2width[width_pred_id]
        all_predictions.append({
            "fold": fold_id,
            "query": row["query"],
            "gold_strategy": f"({gold_depth},{gold_width})",
            "pred_strategy": f"({pred_depth},{pred_width})",
            "gold_depth": gold_depth,
            "pred_depth": pred_depth,
            "gold_width": gold_width,
            "pred_width": pred_width,
            "correct": gold_depth == pred_depth and gold_width == pred_width,
        })

results_df = pd.DataFrame(fold_results)
predictions_df = pd.DataFrame(all_predictions)
display(results_df)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 01 | train_loss=2.6968 | val_loss=2.3266 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 02 | train_loss=2.5354 | val_loss=2.4179 | exact_acc=0.292 | depth_acc=0.458 | width_acc=0.625
Epoch 03 | train_loss=2.4241 | val_loss=2.3525 | exact_acc=0.333 | depth_acc=0.500 | width_acc=0.625
Epoch 04 | train_loss=2.3419 | val_loss=2.3745 | exact_acc=0.292 | depth_acc=0.417 | width_acc=0.625
Epoch 05 | train_loss=2.2507 | val_loss=2.4043 | exact_acc=0.292 | depth_acc=0.417 | width_acc=0.667
Epoch 06 | train_loss=2.1781 | val_loss=2.3903 | exact_acc=0.250 | depth_acc=0.375 | width_acc=0.667
Early stopping at epoch 6.

===== Fold 5/5 =====
Raw train size: 104 | Oversampled train size: 182 | Val size: 23
Oversampled train strategy distribution: {'(3,5)': 10, '(1,2)': 11, '(4,3)': 10, '(1,1)': 35, '(4,5)': 10, '(2,1)': 16, '(2,2)': 10, '(3,1)': 10, '(2,5)': 10, '(4,1)': 10, '(4,2)': 10, '(3,3)': 10, '(1,5)': 10, '(1,3)': 10, '(2,3)': 10}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 01 | train_loss=2.7250 | val_loss=2.3979 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 02 | train_loss=2.6091 | val_loss=2.3751 | exact_acc=0.348 | depth_acc=0.565 | width_acc=0.696
Epoch 03 | train_loss=2.4406 | val_loss=2.4619 | exact_acc=0.304 | depth_acc=0.522 | width_acc=0.609
Epoch 04 | train_loss=2.3712 | val_loss=2.4896 | exact_acc=0.261 | depth_acc=0.565 | width_acc=0.609
Epoch 05 | train_loss=2.2648 | val_loss=2.4089 | exact_acc=0.348 | depth_acc=0.565 | width_acc=0.652
Epoch 06 | train_loss=2.1909 | val_loss=2.3784 | exact_acc=0.304 | depth_acc=0.522 | width_acc=0.696
Epoch 07 | train_loss=2.1317 | val_loss=2.4043 | exact_acc=0.348 | depth_acc=0.565 | width_acc=0.696
Early stopping at epoch 7.


,exact_accuracy,depth_accuracy,width_accuracy,depth_macro_f1,width_macro_f1,strategy_macro_f1,depth_abs_error,width_abs_error,cost_sensitive_error,loss,fold,best_epoch
0,0.346154,0.538462,0.576923,0.347222,0.182927,0.064103,0.769231,0.961538,1.250000,2.407849,1,5
1,0.333333,0.481481,0.555556,0.162500,0.178571,0.041667,0.814815,1.037037,1.333333,2.479907,2,4
2,0.259259,0.444444,0.518519,0.153846,0.175000,0.032634,0.888889,1.074074,1.425926,2.472880,3,4
3,0.333333,0.541667,0.625000,0.175676,0.192308,0.050000,0.708333,0.833333,1.125000,2.326639,4,1
4,0.347826,0.565217,0.695652,0.244318,0.369369,0.051948,0.695652,0.565217,0.978261,2.375111,5,2


## Cross-Validation Summary

In [11]:
metric_cols = [
    "exact_accuracy",
    "depth_accuracy",
    "width_accuracy",
    "depth_macro_f1",
    "width_macro_f1",
    "strategy_macro_f1",
    "depth_abs_error",
    "width_abs_error",
    "cost_sensitive_error",
    "loss",
]

summary = pd.DataFrame({
    "mean": results_df[metric_cols].mean(),
    "std": results_df[metric_cols].std(ddof=1),
})
display(summary)

results_df.to_csv("two_head_frozen_distilbert_oversampling_cv_results.csv", index=False)
predictions_df.to_csv("two_head_frozen_distilbert_oversampling_cv_predictions.csv", index=False)

print("Saved fold metrics to two_head_frozen_distilbert_oversampling_cv_results.csv")
print("Saved validation predictions to two_head_frozen_distilbert_oversampling_cv_predictions.csv")

,mean,std
exact_accuracy,0.323981,0.036824
depth_accuracy,0.514254,0.049704
width_accuracy,0.594330,0.068485
depth_macro_f1,0.216712,0.081194
width_macro_f1,0.219635,0.083954
strategy_macro_f1,0.048070,0.011780
depth_abs_error,0.775384,0.079575
width_abs_error,0.894240,0.205685
cost_sensitive_error,1.222504,0.175771
loss,2.412477,0.065156


Saved fold metrics to two_head_frozen_distilbert_oversampling_cv_results.csv
Saved validation predictions to two_head_frozen_distilbert_oversampling_cv_predictions.csv


## Inspect Predictions

In [12]:
display(predictions_df.head(30))

display(
    predictions_df.groupby(["gold_strategy", "pred_strategy"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

display(
    predictions_df.groupby(["gold_depth", "pred_depth"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(
    predictions_df.groupby(["gold_width", "pred_width"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)


,fold,query,gold_strategy,pred_strategy,gold_depth,pred_depth,gold_width,pred_width,correct
0,1,what continent is australia in,"(1,2)","(1,1)",1,1,2,1,False
1,1,who wrote the book of st. john,"(1,1)","(1,1)",1,1,1,1,True
2,1,what county is rihanna from,"(1,1)","(1,1)",1,1,1,1,True
3,1,who is jojo simmons mother,"(1,1)","(2,1)",1,2,1,1,False
4,1,who created the character of sherlock holmes,"(1,1)","(1,1)",1,1,1,1,True
5,1,who plays lola bunny on the looney tunes show,"(3,1)","(3,3)",3,3,1,3,False
6,1,what instruments does john williams use,"(1,3)","(1,1)",1,1,3,1,False
7,1,where did robin gibb die,"(1,1)","(1,1)",1,1,1,1,True
8,1,who played daniel larusso,"(2,1)","(1,1)",2,1,1,1,False
9,1,who was king or queen after victoria,"(2,1)","(1,1)",2,1,1,1,False


,gold_strategy,pred_strategy,count
0,"(1,1)","(1,1)",40
8,"(2,1)","(1,1)",19
4,"(1,2)","(1,1)",13
15,"(3,1)","(1,1)",9
11,"(2,3)","(1,1)",6
13,"(2,5)","(1,1)",6
7,"(1,5)","(1,1)",4
20,"(4,1)","(1,1)",4
22,"(4,3)","(1,1)",4
5,"(1,3)","(1,1)",3


,gold_depth,pred_depth,count
0,1,1,62
3,2,1,35
5,3,1,12
8,4,1,12
4,2,2,2
1,1,2,1
2,1,3,1
6,3,2,1
7,3,3,1


,gold_width,pred_width,count
0,1,1,74
4,2,1,18
5,3,1,16
7,5,1,14
1,1,2,1
3,1,5,1
2,1,3,1
6,3,5,1
8,5,5,1
